# Lesson 7 — The decoder: self-attention, cross-attention, feed-forward

**Your result:** a Transformer decoder layer that reads an encoder's memory. It will match the decoder inside the installed ACT model, output and gradient. Then you use it to predict a chunk of 16 future robot actions from a short history of real robot states.

Two short exercises and one larger experiment. 

**Sources.** [Attention Is All You Need](https://arxiv.org/abs/1706.03762), sections 3.1 (decoder stack), 3.2.3 (where queries, keys and values come from), 3.3 (feed-forward) and 3.5 (positions). ACT's decoder uses *learned action queries* and reads the encoder's tokens by cross-attention — see the [ACT paper](https://arxiv.org/abs/2304.13705), appendix on architecture, and `ACTDecoderLayer` in the pinned `lerobot/policies/act/modeling_act.py`. ACT's decoder has no causal mask: all 20 actions are predicted at once.

**Where this unit sits.** In Lesson 6 you wrote attention, heads, positions and the encoder layer. **Here we use the package versions of those five functions** (`vla_lab.teaching.attention`), so you can start without your Lesson 6 notebook. Your Lesson 7 functions are wired into the real decoder through its hooks.

# Before you code

This course assumes strong Python skills and basic machine-learning knowledge. We explain PyTorch when it first appears. Read each shape, unit, mask, and parameter definition before you run the check. The reference copy gives a complete solution; the learner copy leaves the implementation for you.

The decoder receives a memory sequence from ACT’s encoder and a learned query for each predicted action time. Each query can attend to the memory and produce one action position. The tensors have these shapes:

| Tensor | Shape |
|---|---|
| decoder queries | $(B,T,E)$ |
| encoder memory | $(B,N,E)$ |
| action head output | $(B,T,D)$ |

Where:
* $B$ - batch size.
* $T$ - chunk horizon: the number of predicted action steps, one query each.
* $N$ - number of memory tokens produced by the encoder.
* $E$ - hidden width of every token.
* $D$ - controller action width ($D=7$).

A causal mask is not automatically required: ACT predicts a whole chunk from the observation memory and learned positions. The key-padding mask instead blocks nonexistent memory tokens. A zero vector is used as the query content before position information is added; the learned position identifies action step $0, 1, \dots, T-1$. For a 20-step chunk, query 0 predicts the first action and query 19 predicts the twentieth. Later control code may use only a prefix and then ask for a new prediction.


In [ ]:
import os
import sys
from collections.abc import Sequence

# Supplied: workspace, offline caches, imports
from pathlib import Path

from vla_lab.teaching.blocks import AttentionOperation, DecoderParts
from vla_lab.teaching.records import TrialConfig, TrialResult


def find_root() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository tree.")


WORKSPACE = find_root()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Set cache paths and offline mode before any data or model library is imported.
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS, DATA = WORKSPACE / "configs", WORKSPACE / ".local-data"
import time

import matplotlib.pyplot as plt
import torch
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTDecoder, ACTDecoderLayer
from torch import nn

import vla_lab.teaching.attention as ref  # package reference: used by checks only
from vla_lab.teaching.attention import (  # your Lesson 6 functions, as package versions
    MultiHeadAttention,
    TransformerDecoder,
    load_lerobot_decoder,
    load_lerobot_decoder_layer,
    sinusoidal_positions,
)
from vla_lab.teaching.diffusion import load_state_action_windows
from vla_lab.teaching.training import masked_action_l1  # package version from the training unit

torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("Workspace:", WORKSPACE)

In [ ]:
# Supplied: a helper that the checks use
def close(actual: torch.Tensor, expected: torch.Tensor, message: str, atol: float = 1e-5) -> None:
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, (
        f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    )
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: encoder memory, queries, and cross-attention

An **encoder** turns the input into a list of *memory* tokens. A **decoder** starts with a list of *query* tokens. In ACT the queries start as zeros: one per future step. Their only information is a learned **query position** vector (one row per future step). In each decoder layer:

1. **Self-attention** lets the queries look at each other.
2. **Cross-attention** lets each query read the memory. Query = decoder token (+ query position). Key = memory token (+ memory position). Value = memory token, with no position.
3. A **feed-forward** block processes every token alone.

Each block adds its result to its input (residual) and normalizes (`LayerNorm`). `LayerNorm` rescales each token's features to zero mean and unit variance, then applies a learned scale and shift.

**Real data used in this unit.** The cells below read real LIBERO `bowl_plate` demonstrations from local files
(no download starts here). Each *window* is 8 consecutive past robot states (8 values each) and the next 16 actions (7 values each). A history that starts before the episode repeats the first state and sets `state_is_pad=True`; a window that runs past the end repeats the last action and sets `action_is_pad=True`. The converted dataset declares a 10 FPS storage label. We select consecutive frame offsets. This label does not establish physical recording intervals, and it does not change the 20 Hz controller or cause action repetition. If the files are missing, the cell tells you the one setup command.

In [ ]:
# Supplied: real windows. 8 states of history; actions start at the current frame.
def read_windows(split: str) -> dict[str, torch.Tensor]:
    return load_state_action_windows(
        CONFIGS,
        DATA,
        "bowl_plate",
        split,
        horizon=16,
        n_obs_steps=8,
        action_start=0,
        drop_n_last_frames=0,
    )


TRAIN, VALID = read_windows("train"), read_windows("validation")
EXAMPLE = int(
    (VALID["state_is_pad"].sum(1) == 3).nonzero()[0]
)  # a window with 3 padded history states
print("train windows:", len(TRAIN["action"]), "validation windows:", len(VALID["action"]))
print(
    "states:",
    tuple(VALID["observation.state"].shape[1:]),
    "actions:",
    tuple(VALID["action"].shape[1:]),
)
print("example history padding:", VALID["state_is_pad"][EXAMPLE].int().tolist())

## Exercise 7.1 — Attention with positions on the query and key only

**Purpose.** Decoder layers use attention two ways: tokens attend to themselves, and tokens attend to encoder memory. One small function handles both and applies the position rule correctly.

**Contract (shapes, units, masks).**

- `attention` is a callable `attention(query, key, value, key_padding_mask) -> (B, Tq, D)` (a `MultiHeadAttention` module).
- `query_input (B, Tq, D)` and `key_value_input (B, Tk, D)`.
- `query_pos` broadcasts to `(B, Tq, D)`; `key_pos` broadcasts to `(B, Tk, D)`. Either may be `None` (no position).
- **Query** = `query_input + query_pos`. **Key** = `key_value_input + key_pos`. **Value** = `key_value_input`, with *no* position.
- `key_padding_mask (B, Tk)` with `True` = padding is passed through unchanged. Return the attention output `(B, Tq, D)`.

**Method and source.** Positions help queries find the right keys, but copying them into the values would add them to the output. This is the placement used by DETR and ACT; the pinned `ACTDecoderLayer` does the same. [Vaswani et al. 2017](https://arxiv.org/abs/1706.03762), sections 3.2.3 and 3.5, describe the three uses of attention.

**Pointers.** Build the three tensors first, handling `None`. Then make one call to `attention`. Self-attention is the case where both inputs are the same tensor.

<details><summary>Detailed hint</summary>

```python
query = query_input if query_pos is None else query_input + query_pos
key = key_value_input if key_pos is None else key_value_input + key_pos
return attention(query, key, key_value_input, key_padding_mask)
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 7.1
def attend_with_positions(
    attention: AttentionOperation,
    query_input: torch.Tensor,
    key_value_input: torch.Tensor,
    query_pos: torch.Tensor | None = None,
    key_pos: torch.Tensor | None = None,
    key_padding_mask: torch.Tensor | None = None,
) -> torch.Tensor:
    """Attention where positions help the match but are not copied into the result.

    ``query_input (B, Tq, D)`` and ``key_value_input (B, Tk, D)``. ``query_pos`` broadcasts to
    ``(B, Tq, D)`` and ``key_pos`` to ``(B, Tk, D)``; either may be ``None``. The query is
    ``query_input + query_pos``, the key is ``key_value_input + key_pos`` and the value is
    ``key_value_input`` with **no** position. Return ``attention(query, key, value, key_padding_mask)``.
    Self-attention passes the same tensor twice; cross-attention passes encoder memory as the second.
    """
    query = query_input if query_pos is None else query_input + query_pos
    key = key_value_input if key_pos is None else key_value_input + key_pos
    return attention(query, key, key_value_input, key_padding_mask)

In [ ]:
# Check: Exercise 7.1
class Recorder:
    """A stand-in attention module that records what it was called with."""

    def __call__(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        self.args = (query, key, value, key_padding_mask)
        return query


recorder = Recorder()
x, memory = torch.randn(2, 4, 8), torch.randn(2, 6, 8)
query_pos, key_pos = torch.randn(1, 4, 8), torch.randn(1, 6, 8)
mask = torch.zeros(2, 6, dtype=torch.bool)
mask[0, 4:] = True
attend_with_positions(recorder, x, memory, query_pos, key_pos, mask)
q, k, v, passed = recorder.args
close(q, x + query_pos, "query must be query_input + query_pos")
close(k, memory + key_pos, "key must be key_value_input + key_pos")
assert torch.equal(v, memory), "The value must be key_value_input with no position."
assert passed is mask or torch.equal(passed, mask), "Pass key_padding_mask through unchanged."
attend_with_positions(recorder, x, x)
q, k, v, passed = recorder.args
assert torch.equal(q, x) and torch.equal(k, x) and torch.equal(v, x) and passed is None, (
    "With no positions and no mask, query, key and value are the input."
)
torch.manual_seed(1)
attention = MultiHeadAttention(8, 2)
close(
    attend_with_positions(attention, x, memory, query_pos, key_pos, mask),
    attention(x + query_pos, memory + key_pos, memory, mask),
    "result with a real attention module",
)
print("Exercise 7.1 passed")

**Use your function.** Self-attention with a query position (as the decoder queries do), then cross-attention to a real state memory. Look at which memory tokens the queries read.

In [ ]:
# Use your function: Exercise 7.1
torch.manual_seed(2)
W = 16
read_states = torch.nn.Linear(8, W)
memory = read_states(VALID["observation.state"][EXAMPLE][None])  # (1, 8, W): eight state tokens
history_pad = VALID["state_is_pad"][EXAMPLE][None]
queries = torch.zeros(1, 16, W)  # ACT-style zero queries
query_pos = torch.randn(1, 16, W)  # stand-in for learned query positions
memory_pos = sinusoidal_positions(8, W)[None]
cross = MultiHeadAttention(W, 4)
with torch.no_grad():
    result = attend_with_positions(cross, queries, memory, query_pos, memory_pos, history_pad)
weights = cross.last_weights[0].mean(0)  # average over heads: (16 queries, 8 memory tokens)
plt.imshow(weights, cmap="viridis")
plt.colorbar()
plt.xlabel("memory token (state history, oldest first)")
plt.ylabel("query (future step)")
plt.title("Cross-attention weights (random weights). Padded history gets 0")
plt.show()
print("weight on padded history tokens:", float(weights[:, history_pad[0]].sum()))

## Exercise 7.2 — Decoder layer: self-attention, cross-attention, feed-forward

**Purpose.** Assemble the three blocks of a decoder layer with residual connections and norms. This is the layer ACT uses to turn learned queries into actions.

**Contract (shapes, units, masks).**

- `layer` has `self_attn`, `cross_attn` (callables like Exercise 7.1's `attention`), `linear1`, `linear2`, `norm1`, `norm2`, `norm3`, `dropout`, `dropout1`, `dropout2`, `dropout3`, `activation` and the bool `pre_norm`.
- `x (B, Tq, D)` queries; `memory (B, Tm, D)`; `query_pos` broadcasts to `(B, Tq, D)` or is `None`; `memory_pos` broadcasts to `(B, Tm, D)` or is `None`; `memory_key_padding_mask (B, Tm)` with `True` = padding, or `None`.
- Self-attention: `attend_with_positions(layer.self_attn, x, x, query_pos, query_pos)`; no mask. Cross-attention: query from `x`, key/value from `memory`, with `query_pos`, `memory_pos` and `memory_key_padding_mask`. **Call your Exercise 7.1 function for both.**
- Post-norm (`pre_norm=False`): `x = norm1(x + dropout1(self_attn))`; `x = norm2(x + dropout2(cross_attn))`; `x = norm3(x + dropout3(ff))`, with `ff(x) = linear2(dropout(activation(linear1(x))))`.
- Pre-norm: each norm goes before its block (`norm1` before self-attention, `norm2` before cross-attention, `norm3` before the feed-forward) and none after the add.
- Return `(B, Tq, D)`.

**Method and source.** Decoder layer: [Vaswani et al. 2017](https://arxiv.org/abs/1706.03762), section 3.1 (three sub-layers, each with a residual connection and layer normalization). ACT's `ACTDecoderLayer` (pinned) is this layer without a causal mask.

**Pointers.** Copy the structure of the encoder layer from Lesson 6 and insert the cross-attention block between self-attention and the feed-forward block. Keep a `skip` tensor before each block.

<details><summary>Detailed hint</summary>

Post-norm version:
```python
skip = x
x = attend_with_positions(layer.self_attn, x, x, query_pos, query_pos)
x = layer.norm1(skip + layer.dropout1(x))
skip = x
x = attend_with_positions(layer.cross_attn, x, memory, query_pos, memory_pos, memory_key_padding_mask)
x = layer.norm2(skip + layer.dropout2(x))
skip = x
x = layer.linear2(layer.dropout(layer.activation(layer.linear1(x))))
return layer.norm3(skip + layer.dropout3(x))
```
For pre-norm, apply `norm1/2/3` to `x` just before each block instead and drop the norm after each add.

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 7.2
def decoder_layer_forward(
    layer: DecoderParts,
    x: torch.Tensor,
    memory: torch.Tensor,
    query_pos: torch.Tensor | None = None,
    memory_pos: torch.Tensor | None = None,
    memory_key_padding_mask: torch.Tensor | None = None,
) -> torch.Tensor:
    """One decoder layer (section 3.1): self-attention, cross-attention, feed-forward.

    ``x (B, Tq, D)`` are the queries, ``memory (B, Tm, D)`` is the encoder output.
    Self-attention uses ``query_pos`` on both sides; cross-attention uses ``query_pos`` on
    the query and ``memory_pos`` on the memory key (``attend_with_positions``). Values never
    receive a position. Each of the three blocks has a residual connection and a norm, placed as
    in ``encoder_layer_forward`` (post-norm by default, ``layer.pre_norm`` for pre-norm), with
    ``norm1``/``dropout1`` after self-attention, ``norm2``/``dropout2`` after cross-attention and
    ``norm3``/``dropout3`` after the feed-forward block.
    Layer attributes: ``self_attn``, ``cross_attn``, ``linear1``, ``linear2``, ``norm1..3``,
    ``dropout``, ``dropout1..3``, ``activation``, ``pre_norm``.
    """
    skip = x
    if layer.pre_norm:
        x = layer.norm1(x)
    x = attend_with_positions(layer.self_attn, x, x, query_pos, query_pos)
    x = skip + layer.dropout1(x)
    if layer.pre_norm:
        skip = x
        x = layer.norm2(x)
    else:
        x = layer.norm1(x)
        skip = x
    x = attend_with_positions(
        layer.cross_attn, x, memory, query_pos, memory_pos, memory_key_padding_mask
    )
    x = skip + layer.dropout2(x)
    if layer.pre_norm:
        skip = x
        x = layer.norm3(x)
    else:
        x = layer.norm2(x)
        skip = x
    x = layer.linear2(layer.dropout(layer.activation(layer.linear1(x))))
    x = skip + layer.dropout3(x)
    if not layer.pre_norm:
        x = layer.norm3(x)
    return x

In [ ]:
# Check: Exercise 7.2
D_, H_, FF_ = 16, 4, 32


def config(pre: bool) -> ACTConfig:
    return ACTConfig(
        dim_model=D_,
        n_heads=H_,
        dim_feedforward=FF_,
        dropout=0.0,
        pre_norm=pre,
        chunk_size=4,
        n_action_steps=4,
        input_features={},
        output_features={},
    )


for pre in (False, True):
    torch.manual_seed(3)
    oracle = ACTDecoderLayer(config(pre)).eval()
    mine = ref.DecoderLayer(D_, H_, FF_, pre_norm=pre, forward_fn=decoder_layer_forward).eval()
    load_lerobot_decoder_layer(mine, oracle)
    x = torch.randn(3, 4, D_, requires_grad=True)
    memory = torch.randn(3, 7, D_, requires_grad=True)
    query_pos, memory_pos = torch.randn(1, 4, D_), torch.randn(1, 7, D_)
    out = mine(x, memory, query_pos, memory_pos)
    expected = oracle(
        x.transpose(0, 1),
        memory.transpose(0, 1),
        decoder_pos_embed=query_pos.transpose(0, 1),
        encoder_pos_embed=memory_pos.transpose(0, 1),
    ).transpose(0, 1)
    close(out, expected, f"layer output (pre_norm={pre}) differs from ACTDecoderLayer")
    out.square().sum().backward()
    x2, memory2 = (t.detach().clone().requires_grad_(True) for t in (x, memory))
    oracle(
        x2.transpose(0, 1),
        memory2.transpose(0, 1),
        decoder_pos_embed=query_pos.transpose(0, 1),
        encoder_pos_embed=memory_pos.transpose(0, 1),
    ).transpose(0, 1).square().sum().backward()
    close(x.grad, x2.grad, f"gradient of the queries (pre_norm={pre})", atol=1e-4)
    close(memory.grad, memory2.grad, f"gradient of the memory (pre_norm={pre})", atol=1e-4)
    close(
        mine.cross_attn.k_proj.weight.grad,
        oracle.multihead_attn.in_proj_weight.grad[D_ : 2 * D_],
        "cross-attention key gradient",
        atol=1e-4,
    )
# memory padding: the pinned layer has no mask, so check the behaviour directly.
mask = torch.zeros(3, 7, dtype=torch.bool)
mask[:, 5:] = True
base = mine(x.detach(), memory.detach(), query_pos, memory_pos, mask)
changed = memory.detach().clone()
changed[:, 5:] = 40.0
close(
    mine(x.detach(), changed, query_pos, memory_pos, mask),
    base,
    "padded memory must not change the output",
    atol=1e-4,
)
assert not torch.allclose(
    mine(x.detach(), memory.detach(), query_pos, memory_pos), base, atol=1e-4
), "The mask must change the result when padded memory is real data."
print("Exercise 7.2 passed")

**Use your function.** Stack two of your layers into a real decoder and run ACT-style zero queries against a real state memory. Compare with the installed ACT decoder stack.

In [ ]:
# Use your function: Exercise 7.2
D_, H_, FF_ = 16, 4, 32
torch.manual_seed(4)
pinned = ACTDecoder(
    ACTConfig(
        dim_model=D_,
        n_heads=H_,
        dim_feedforward=FF_,
        n_decoder_layers=2,
        dropout=0.0,
        chunk_size=16,
        n_action_steps=16,
        input_features={},
        output_features={},
    )
).eval()
yours = TransformerDecoder(2, D_, H_, FF_, forward_fn=decoder_layer_forward).eval()
load_lerobot_decoder(yours, pinned)
embed = torch.nn.Linear(8, D_)
memory = embed(VALID["observation.state"][:6])  # (6, 8, D): real state histories
query_pos, memory_pos = torch.randn(1, 16, D_), sinusoidal_positions(8, D_)[None]
queries = torch.zeros(6, 16, D_)
with torch.no_grad():
    out = yours(queries, memory, query_pos, memory_pos)
    theirs = pinned(
        queries.transpose(0, 1),
        memory.transpose(0, 1),
        decoder_pos_embed=query_pos.transpose(0, 1),
        encoder_pos_embed=memory_pos.transpose(0, 1),
    ).transpose(0, 1)
print("largest difference from the installed ACT decoder:", float((out - theirs).abs().max()))

## Experiment — what does the decoder read?

**Question.** A decoder predicts 16 future actions from a history of real robot states. Which parts of its input matter: the length of the history, the positions on the memory, or the memory padding mask?

**Task (real data).** Memory: up to 8 robot states (oldest first), each projected to 32 features. Queries: 16 zero vectors plus learned query positions (one row per future step, as in ACT). Output: 16 actions of 7 values in $[-1, 1]$, min–max scaled per action value with training extrema:

$$\tilde{a}_d=2\,\frac{a_d-a_d^{\min}}{a_d^{\max}-a_d^{\min}}-1$$

Where:
* $a_d$ - raw action value $d$, $1 \le d \le 7$.
* $a_d^{\min}$, $a_d^{\max}$ - smallest and largest value of action $d$ over training windows.
* $\tilde{a}_d$ - the scaled target.

Loss: masked L1 over real (non-padded) actions — the function from the training unit. States before the episode start are padded copies of the first state.

**Fixed for every trial.** The windows above (validation episodes are never used for updates); seed `0` for weights, seed `1` for batch draws; **200 updates**, batch 64, AdamW `1e-3`; width 32, 4 heads. Your Exercises 7.1 and 7.2 functions run inside every decoder layer.

**Allowed changes (per trial).** `k` (use only the newest `k` states, 1–8), `mem_pos` (learned memory positions on or off), `mask` (pass the history padding mask to cross-attention or not), `layers` (1 or 2).

**Measures.** `val_l1`: masked L1 on all validation windows. `val_l1_padhist`: the same on windows whose history contains padding. `pad_sensitivity`: mean change of the output when the padded history states are replaced by random values.

**Prediction first.** Write two sentences in the next cell before you run anything: which setting will hurt `val_l1` most, and which will change `pad_sensitivity`?

**Keep failed trials.** Every trial is appended to `TRIALS` as a `TrialResult` dict: `config` holds the trial's `TrialConfig` settings, `status` is `"ok"` or `"failed: <error>"`, and the measures above are extra keys of the same row. Never delete rows. No improvement is a valid result.

In [ ]:
# Run this cell to start the larger training experiment.


In [ ]:
# Write your prediction in the Markdown cell above before you run the experiment.


In [ ]:
# Supplied: trial table helper
import pandas as pd


def show_trials(trials: Sequence[TrialResult], columns: Sequence[str]) -> pd.DataFrame:
    """Print every trial that was run, including failed ones, and return a table."""
    table = pd.DataFrame(
        [{**t["config"], **{c: t.get(c) for c in columns}, "status": t["status"]} for t in trials]
    )
    print(table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    return table

In [ ]:
# Supplied: harness. Your decoder_layer_forward and attend_with_positions are in the real path.
STEPS, BATCH, LR, SEED_WEIGHTS, SEED_BATCH, WIDTH = 200, 64, 1e-3, 0, 1, 32


class HistoryDecoder(nn.Module):
    def __init__(self, layers: int = 1, mem_pos: bool = True) -> None:
        super().__init__()
        self.inp = nn.Linear(8, WIDTH)
        self.mem_pos = nn.Embedding(8, WIDTH) if mem_pos else None
        self.query_pos = nn.Embedding(16, WIDTH)
        self.decoder = TransformerDecoder(
            layers, WIDTH, 4, 2 * WIDTH, forward_fn=decoder_layer_forward
        )
        self.out = nn.Linear(WIDTH, 7)

    def forward(
        self, states: torch.Tensor, state_pad: torch.Tensor, k: int = 8, use_mask: bool = True
    ) -> torch.Tensor:
        states, state_pad = states[:, -k:], state_pad[:, -k:]
        memory_pos = self.mem_pos.weight[None, -k:] if self.mem_pos is not None else None
        queries = torch.zeros(states.shape[0], 16, WIDTH)
        hidden = self.decoder(
            queries,
            self.inp(states),
            self.query_pos.weight[None],
            memory_pos,
            state_pad if use_mask else None,
        )
        return self.out(hidden)


BASELINE = {"k": 8, "mem_pos": True, "mask": True, "layers": 1}
TRIALS = []


def run_trial(config: TrialConfig) -> dict[str, float | int]:
    started = time.perf_counter()
    torch.manual_seed(SEED_WEIGHTS)
    model = HistoryDecoder(config["layers"], config["mem_pos"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    draws = torch.Generator().manual_seed(SEED_BATCH)
    for _ in range(STEPS):
        pick = torch.randint(0, len(TRAIN["action"]), (BATCH,), generator=draws)
        prediction = model(
            TRAIN["observation.state"][pick],
            TRAIN["state_is_pad"][pick],
            config["k"],
            config["mask"],
        )
        loss = masked_action_l1(prediction, TRAIN["action"][pick], TRAIN["action_is_pad"][pick])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        S, P, A, AP = (
            VALID[key] for key in ("observation.state", "state_is_pad", "action", "action_is_pad")
        )
        has = P[:, -config["k"] :].any(1)
        shaken = torch.where(
            P[has].unsqueeze(-1),
            torch.randn(S[has].shape, generator=torch.Generator().manual_seed(5)) * 3,
            S[has],
        )
        base = model(S[has], P[has], config["k"], config["mask"])
        return {
            "val_l1": masked_action_l1(model(S, P, config["k"], config["mask"]), A, AP).item(),
            "val_l1_padhist": masked_action_l1(base, A[has], AP[has]).item()
            if has.any()
            else float("nan"),
            "pad_sensitivity": (model(shaken, P[has], config["k"], config["mask"]) - base)
            .abs()
            .mean()
            .item()
            if has.any()
            else 0.0,
            "seconds": time.perf_counter() - started,
        }


def record(config: TrialConfig) -> None:
    config = {**BASELINE, **config}
    try:
        TRIALS.append({"config": config, "status": "ok", **run_trial(config)})
    except Exception as error:  # a failed trial is evidence too
        TRIALS.append({"config": config, "status": f"failed: {type(error).__name__}: {error}"})


record(BASELINE)
COLUMNS = ["val_l1", "val_l1_padhist", "pad_sensitivity", "seconds"]
show_trials(TRIALS, COLUMNS);

**Your trials.** Add at least two trials that each change one setting. `k=1` has no padded history, so its `pad_sensitivity` is 0 by construction: read it with that in mind.

In [ ]:
# Reference choice of trials: one change per trial.
MY_TRIALS = [{"k": 1}, {"k": 4}, {"mem_pos": False}, {"mask": False}, {"layers": 2}]
for trial in MY_TRIALS:
    record(trial)
show_trials(TRIALS, COLUMNS);

In [ ]:
# Check: the protocol was followed.
assert TRIALS[0]["config"] == BASELINE, "The first trial must be the baseline."
assert len(TRIALS) >= 3, "Run the baseline and at least two trials of your own."
assert all(set(t["config"]) == set(BASELINE) for t in TRIALS), (
    "Only k, mem_pos, mask and layers may change."
)
ok = [t for t in TRIALS if t["status"] == "ok"]
assert all(t["status"] == "ok" or t["status"].startswith("failed") for t in TRIALS)
assert all(torch.isfinite(torch.tensor([t["val_l1"], t["pad_sensitivity"]])).all() for t in ok), (
    "Measures must be finite."
)
print(
    f"{len(TRIALS)} trials kept ({len(ok)} ok). Budget: {STEPS} updates, batch {BATCH}, seeds {SEED_WEIGHTS}/{SEED_BATCH}."
)

In [ ]:
# Use your function: plot every trial
ok = [t for t in TRIALS if t["status"] == "ok"]
labels = [
    ", ".join(f"{k}={v}" for k, v in t["config"].items() if v != BASELINE[k]) or "baseline"
    for t in ok
]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, column in zip(axes, ["val_l1", "val_l1_padhist", "pad_sensitivity"], strict=True):
    axis.bar(range(len(ok)), [t[column] for t in ok])
    axis.set_xticks(range(len(ok)))
    axis.set_xticklabels(labels, rotation=30, ha="right")
    axis.set_title(column)
plt.tight_layout()
plt.show()

**Explain (2–4 sentences, edit this cell).** Which setting changed `val_l1` the most? Which one changed only `pad_sensitivity`? Did your prediction hold? Remember that the target is a future action chunk that also depends on the camera images, which this model never sees, so a large remaining error is expected.

*Optional extension:* replace a trial's decoder layers by the package `DecoderLayer` default (no learner hooks) and confirm the table is identical for the baseline. Keep all rows.

## Optional: your decoder layer with real ACT weights

This copies the **prepared** ACT checkpoint's decoder weights (`act-bowl-plate-probe-200`) into a decoder built from *your* `decoder_layer_forward`, then compares it with the installed ACT decoder on a real state memory. The memory has only the latent and state tokens (no camera tokens), so the outputs are not the policy's actions; the check shows that your layer reproduces a real trained decoder. The workload is a few validation frames. Run the next cell to run it.

In [ ]:
from lerobot.policies.act import ACTPolicy

from vla_lab.teaching.act_model import load_act_samples

checkpoint = DATA / "runs/act-bowl-plate-probe-200/checkpoint-000200"
if not (checkpoint / "model.safetensors").is_file():
    raise FileNotFoundError(f"Prepared ACT checkpoint missing: {checkpoint}")
config = ACTConfig.from_pretrained(checkpoint, local_files_only=True)
config.device = "cpu"
pinned = ACTPolicy.from_pretrained(
    checkpoint, config=config, strict=True, local_files_only=True
).model.eval()
yours = TransformerDecoder(
    config.n_decoder_layers,
    config.dim_model,
    config.n_heads,
    config.dim_feedforward,
    forward_fn=decoder_layer_forward,
).eval()
load_lerobot_decoder(yours, pinned.decoder)
samples = load_act_samples(
    CONFIGS, DATA, "bowl_plate", "validation", count=4, chunk_size=config.chunk_size, image_size=32
)
with torch.no_grad():
    latent = pinned.encoder_latent_input_proj(torch.zeros(4, config.latent_dim))
    state = pinned.encoder_robot_state_input_proj(samples["observation.state"])
    memory = torch.stack([latent, state], dim=1)  # (4, 2, D)
    memory_pos = pinned.encoder_1d_feature_pos_embed.weight[None]  # (1, 2, D)
    query_pos = pinned.decoder_pos_embed.weight[None]  # (1, T, D)
    queries = torch.zeros(4, config.chunk_size, config.dim_model)
    mine = yours(queries, memory, query_pos, memory_pos)
    theirs = pinned.decoder(
        queries.transpose(0, 1),
        memory.transpose(0, 1),
        decoder_pos_embed=query_pos.transpose(0, 1),
        encoder_pos_embed=memory_pos.transpose(0, 1),
    ).transpose(0, 1)
    print("largest difference from the installed ACT decoder:", float((mine - theirs).abs().max()))
    print("decoded action shape:", tuple(pinned.action_head(mine).shape))

## What you built

- `attend_with_positions` and `decoder_layer_forward`, checked for output and gradient against the pinned ACT decoder layer, plus a behaviour check for memory padding (which the pinned layer does not have).
- Evidence from real data about which decoder inputs matter, with every trial kept.

**Next:** Lesson 8 assembles the whole ACT model: camera, state and latent tokens, the posterior encoder, action queries and the head. It uses the package versions of the Lesson 6 and Lesson 7 functions.